# 🧱 16 · 3D convolutional networks for subtomogram classification
### Learned features for millions of particles: classify, subdivide, re-balance, compress and transfer

**BiologyToResearch** · Part D — Deep learning on subtomograms (2017–2024) · Module 16 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/16_3d_cnn_classification.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 4–6 min to run everything); a GPU is used automatically if present |
| **Core frameworks** | Python 3, NumPy, SciPy, scikit-learn, PyTorch, Matplotlib |
| **Prerequisites** | Module 05 (PyTorch, training loops, CNNs, data splits), Module 07 (missing wedge), Module 09 (simulation); Module 14 (averaging) helps for Section 4 |
| **Estimated time** | 150 min (reading + running + ✏️ exercises) |
| **Lab papers covered** | Xu *et al.* 2017 (deep-learning subdivision); Che *et al.* 2018 (DSRF3D-v2 / RB3D / CB3D); Gao *et al.* 2020/2021 (Dilated-DenseNet); Luo *et al.* 2019 (imbalanced data); Guo *et al.* 2018 (model compression); Jain, Li & Xu 2024 (video-model transfer); Gubins *et al.* SHREC'19/'20 (benchmarks); Uddin *et al.* 2021 (cited only) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments here are small, controlled *toy versions* of published methods, designed to teach the core idea of each
> paper on a laptop CPU in minutes. They do **not** reproduce the papers' architectures at full size, their datasets
> or their numbers. **All molecules and data are generated by code** (nothing is downloaded): realistic stand-ins with
> the size and symmetry of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a 3D convolution computes, **implement** one by hand, and **calculate** the receptive field of a
   stack of ordinary and *dilated* convolutions.
2. **Build and train** two small 3D CNNs — a VGG-style *DSRF3D-lite* and a residual *RB3D-lite* — on simulated
   subtomograms with a missing wedge, using rotation augmentation that keeps the wedge physically realistic.
3. **Evaluate** a classifier with a confusion matrix, per-class recall and accuracy at two noise levels, and **explain**
   why real data must be split by tomogram.
4. **Implement** the *subdivision* approach (Xu *et al.* 2017): pre-softmax outputs → k-means → cluster averages, and
   **show** that it recovers a molecule the network never saw.
5. **Compare** three remedies for class imbalance (re-weighting, focal loss, mixup) with macro-F1 and G-mean.
6. **Compress** a network by knowledge distillation with temperature *T* = 5 and **measure** its speed/accuracy trade-off.
7. **Inflate** 2D convolution kernels into 3D, **verify** the "boring video" property, and **use** it to transfer a
   2D-pretrained network to 3D subtomograms with only a handful of labels.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, import tools, load the toolkit | 5 min |
| 1 · 3D convolutions and receptive fields | a convolution by hand; why dilation enlarges what a neuron "sees" | 20 min |
| 2 · A simulated subtomogram dataset | six molecules, missing wedge, two noise levels, wedge-safe augmentation | 15 min |
| 3 · Two architectures | train DSRF3D-lite and RB3D-lite; confusion matrices | 25 min |
| 4 · Subdivision | CNN features → k-means → averages; recover an unseen molecule | 20 min |
| 5 · Rare molecules | class imbalance: weighting, focal loss, mixup | 15 min |
| 6 · Smaller and faster | knowledge distillation with *T* = 5 | 15 min |
| 7 · From 2D to 3D | kernel inflation, the idea behind video-model transfer | 20 min |
| 8 · Benchmarks and pitfalls | SHREC, splitting by tomogram, molecular weight | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A single cryo-ET session on a bacterial or neuronal cell can yield **hundreds of thousands** of subtomograms — small
3D boxes, each (hopefully) centred on one macromolecule. Before anyone can average them into structures, someone must
answer, box by box: *is this a ribosome, a proteasome, RuBisCO… or something nobody has catalogued?* Classical
alignment-based sorting (Modules 12–15) answers this beautifully but costs seconds to minutes **per pair** of
subtomograms — hopeless at the million-particle scale.

From 2017 on, the Xu lab showed that **3D convolutional neural networks (3D CNNs)**, trained on simulated
subtomograms, can sort particles in about a millisecond each. That speed changed the workflow: a CNN first
*subdivides* a huge, mixed dataset into small, more homogeneous groups, and the slow, careful averaging methods then
work inside each group. This notebook builds that workflow in miniature — and examines the practical problems that
follow: rare molecules, slow models, and too few labelled examples.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell checks the hardware: everything here runs on a
CPU; a GPU only makes it faster.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook is designed for a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix, f1_score

SEED = 0                                   # one seed → identical results on every run
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

The helpers below are copied verbatim from [`course_toolkit.py`](../course_toolkit.py) so that this notebook runs on
its own: building zoo maps (Module 00–01), rotations (Module 02), the missing wedge (Module 07) and the fast
subtomogram simulator (Module 09). Just run the cell.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def simulate_subtomogram(template: np.ndarray, rng: np.random.Generator, snr: float = 0.1,
                         tilt_max_deg: float | None = 60.0, max_shift: float = 0.0,
                         R: np.ndarray | None = None):
    """Randomly rotate/shift a template, impose a missing wedge and add noise.

    Returns (subtomogram, R, shift_xyz).  This 'direct' simulation (Fourier mask +
    white noise) is the fast approximation used by many Xu-lab benchmarks.
    """
    R = random_rotation(rng) if R is None else R
    t = rng.uniform(-max_shift, max_shift, 3) if max_shift > 0 else np.zeros(3)
    v = rotate_volume(template, R, t)
    if tilt_max_deg is not None:
        v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng), R, t
print("🧰 toolkit loaded")

### The cast: molecular templates at 15 Å per voxel

The zoo maps have 10 Å (1 nm) voxels. A yeast 80S ribosome is ~30 nm across, so a 24³ box of 1 nm voxels would cut
it. To keep the CNNs small enough for a CPU we **bin** the maps to **15 Å (1.5 nm) voxels**: a 24³ box then spans
36 nm, enough for every molecule we use. (The papers in this module used 0.92–1.37 nm voxels and 28³–40³ boxes.)

| role in this notebook | PDB | molecule | mass |
|---|---|---|---|
| 6 training classes | 4V4R, 4V7R | bacterial 70S and yeast 80S ribosomes | 2.3, 5.9 MDa |
| | 3J9I, 6UTJ, 4CR2 | 20S proteasome, singly capped proteasome, yeast 26S proteasome | 0.66, 0.84, 1.3 MDa |
| | 1BXN | RuBisCO | 0.28 MDa |
| never shown to the network (Section 4) | 1QVR | ClpB disaggregase — our "unknown" | ≈ 0.3 MDa |
| a *different* task for 2D pre-training (Section 7) | 5MRC, 4V94, 2CG9, 1U6G, 3D2F, 3CF3 | mitoribosome, TRiC (×2), Hsp90, ligase, Hsp70 complex, p97 | — |

Notice that the classes form **biological families**: two ribosomes, three proteasome assemblies. Classifying within a
family is much harder than between families — keep that in mind when you read the confusion matrices.

In [ ]:
# ── Load the templates (binned to 15 Å voxels, 24³ boxes) ──
VOXEL_A = 15.0          # Å per voxel after binning
BOX = 24                # voxels per side → 36 nm box
CLASSES = ["4v4r", "4v7r", "3j9i", "6utj", "4cr2", "1bxn"]
SHORT = {"4v4r": "70S ribo", "4v7r": "80S ribo", "3j9i": "20S prot", "6utj": "capped prot",
         "4cr2": "26S prot", "1bxn": "RuBisCO", "1qvr": "ClpB"}
UNKNOWN = "1qvr"
OTHERS_2D = ["5mrc", "4v94", "2cg9", "1u6g", "3d2f", "3cf3"]


def make_template(pid: str, box: int = BOX, voxel_A: float = VOXEL_A) -> np.ndarray:
    """Zoo map → box³ template at voxel_A Å/voxel (prepared at 10 Å, then resampled), scaled to [0, 1]."""
    raw, vs, src = zoo_map(pid)
    n10 = int(round(box * voxel_A / 10.0))                 # 36 voxels of 10 Å = the same 36 nm box
    v = ndimage.zoom(prepare_map(raw, size=n10, sigma=1.0), box / n10, order=1)
    v = np.clip(v, 0, None)
    print(f"  {pid.upper()} {ZOO[pid]['name']:26s} ← {src}")
    return (v / v.max()).astype(np.float32)


t0 = time.time()
TEMPLATES = {pid: make_template(pid) for pid in CLASSES + [UNKNOWN] + OTHERS_2D}
print(f"{len(TEMPLATES)} templates of shape {TEMPLATES['4v4r'].shape} in {time.time() - t0:.1f} s")

---
## 1 · 3D convolutions and receptive fields

### 📍 Where we are
Module 05 trained your first tiny 3D CNN; Modules 12–15 sorted particles by **aligning** them — accurate, but slow. Here
we open the CNN box properly: what one 3D convolution computes, how far a neuron deep in the network can "see", and how
the lab's networks were designed for noisy, wedge-distorted subtomograms.

🧮 **Refresher (Module 05).** A CNN is a stack of *layers*; each layer applies learnable filters, then a nonlinearity
such as ReLU (max(0, ·)). Training repeats: forward pass → loss → backpropagation (gradients) → optimizer step. We keep
training, validation and test data separate.

### 1.1 What a 3D convolution computes

A **3D convolution** slides a small cube of weights $w$ (the *kernel*, e.g. 3 × 3 × 3 voxels) over the volume $v$ and,
at every position, takes a weighted sum of the voxels underneath:

$$o[z, y, x] \;=\; b + \sum_{a=0}^{2}\sum_{b'=0}^{2}\sum_{c=0}^{2} w[a, b', c]\; v[z+a,\; y+b',\; x+c].$$

*In words:* each output voxel is a "local template match" between the kernel and a 3 × 3 × 3 neighbourhood. The *same*
27 weights are used everywhere (**weight sharing**), so a ribosome is detected the same way wherever it sits in the box.
(Strictly this is a *cross-correlation*: deep-learning libraries do not flip the kernel. It makes no difference for
learning.) A layer with $C_{in}$ input and $C_{out}$ output channels has $C_{in}\,C_{out}\,27 + C_{out}$ parameters.

In [ ]:
# ── 1.1 A 3D convolution by hand, checked against PyTorch ──
def conv3d_by_hand(vol: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """'Valid' 3D cross-correlation written with NumPy: every output = Σ kernel × neighbourhood."""
    windows = np.lib.stride_tricks.sliding_window_view(vol, kernel.shape)   # [z', y', x', 3, 3, 3]
    return np.einsum("zyxabc,abc->zyx", windows, kernel)


example, _, _ = simulate_subtomogram(TEMPLATES["4v4r"], np.random.default_rng(1), snr=0.1)
example = normalize(example)
k_mean = np.full((3, 3, 3), 1 / 27, np.float32)                       # local average (smoothing)
k_lap = np.zeros((3, 3, 3), np.float32); k_lap[1, 1, 1] = 6           # Laplacian: centre minus its 6 neighbours
for dz, dy, dx in [(0, 1, 1), (2, 1, 1), (1, 0, 1), (1, 2, 1), (1, 1, 0), (1, 1, 2)]:
    k_lap[dz, dy, dx] = -1

by_hand = conv3d_by_hand(example, k_mean)
by_torch = F.conv3d(torch.tensor(example)[None, None], torch.tensor(k_mean)[None, None])[0, 0].numpy()
print("output shape:", by_hand.shape, "(a 'valid' convolution loses one voxel on each side)")
print(f"max |by hand − PyTorch| = {np.abs(by_hand - by_torch).max():.2e}")
assert np.allclose(by_hand, by_torch, atol=1e-4), "the hand-made convolution should match nn.functional.conv3d"

conv = nn.Conv3d(16, 32, kernel_size=3)
print(f"nn.Conv3d(16, 32, 3) has {sum(p.numel() for p in conv.parameters()):,} parameters "
      f"(16·32·27 + 32 = {16 * 32 * 27 + 32:,}); a fully connected layer from 24³ voxels to 32 units would need "
      f"{24**3 * 32 + 32:,}.")

fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
mid = example.shape[0] // 2
smooth2 = conv3d_by_hand(by_hand, k_mean)                              # two mean layers in a row
panels = [(example[mid], "noisy ribosome (SNR 0.1)"), (by_hand[mid - 1], "* mean kernel (smooths)"),
          (conv3d_by_hand(example, k_lap)[mid - 1], "* Laplacian (edges… of noise)"),
          (conv3d_by_hand(smooth2, k_lap)[mid - 3], "mean → mean → Laplacian")]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img); ax.set_title(title, fontsize=8); ax.axis("off")
fig.suptitle("Hand-designed 3×3×3 kernels, central xy slice (1 voxel = 1.5 nm)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Ten lines of NumPy reproduce PyTorch's `conv3d` exactly. The mean kernel averages noise away
(at the cost of blur). The Laplacian responds to *changes* of density — but applied directly to a noisy box it mostly
finds the edges of noise. **Stacking** kernels fixes this: two smoothing layers followed by a Laplacian outline the
particle. That is the essence of a deep CNN — except that a CNN does not use hand-designed kernels: it **learns**
thousands of them, chosen by gradient descent to make the classification loss small.

### 1.2 Receptive field: how much of the particle can one neuron see?

The **receptive field** (RF) of a neuron is the region of the input that can influence it. One 3 × 3 × 3 layer sees 3
voxels; stacking layers widens the view. For layers with kernel size $k_l$, dilation $d_l$ and stride $s_l$:

$$\text{RF}_l = \text{RF}_{l-1} + (k_l - 1)\, d_l\, J_{l-1}, \qquad J_l = J_{l-1}\, s_l, \qquad \text{RF}_0 = J_0 = 1 .$$

*In words:* each layer adds $(k-1)$ "steps" to the field of view, and each step is as long as the current *jump* $J$
(how many input voxels separate neighbouring neurons, multiplied by every stride/pooling). A **dilated** convolution
(dilation $d$) spreads its 3 × 3 × 3 taps $d$ voxels apart, so it sees $2d+1$ voxels with only 27 weights.

A 70S ribosome is ≈ 17 voxels across at 15 Å/voxel. Let us measure the receptive field of three stacked convolutions
directly: we ask PyTorch for the **gradient** of the central output voxel with respect to every input voxel — any voxel
with a non-zero gradient is inside the receptive field.

In [ ]:
# ── 1.2 Measure receptive fields with gradients: ordinary vs dilated convolutions ──
def receptive_field(layers):
    """RF size (voxels) of a stack described as [(kernel, stride, dilation), ...]."""
    rf, jump = 1, 1
    for k, s, d in layers:
        rf, jump = rf + (k - 1) * d * jump, jump * s
    return rf


def measured_rf_map(dilations, size=33):
    """|∂ output(centre) / ∂ input| for 3 stacked 3×3×3 convolutions with all-equal weights."""
    stack = nn.Sequential(*[nn.Conv3d(1, 1, 3, padding=d, dilation=d, bias=False) for d in dilations])
    for layer in stack:
        nn.init.constant_(layer.weight, 1 / 27)
    x = torch.zeros(1, 1, size, size, size, requires_grad=True)
    c = size // 2
    stack(x)[0, 0, c, c, c].backward()
    return x.grad[0, 0].abs().numpy()


fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, dil in zip(axes, [(1, 1, 1), (1, 2, 4)]):
    g = measured_rf_map(dil)
    width = np.ptp(np.argwhere(g > 0)[:, 2]) + 1
    formula = receptive_field([(3, 1, d) for d in dil])
    ax.imshow(np.log10(g[g.shape[0] // 2] + 1e-6), cmap="viridis", vmin=-6)
    ax.set_title(f"dilations {dil}: RF = {width} voxels\n(formula {formula}; {width * VOXEL_A / 10:.1f} nm), 81 weights",
                 fontsize=9)
    ax.set_xlabel("x (voxels)"); ax.set_ylabel("y (voxels)")
fig.suptitle("Which input voxels influence the central output? (log₁₀ |gradient|, central slice)", fontsize=10)
plt.tight_layout(); plt.show()

vgg_layers = [(3, 2, 1), (3, 1, 1), (2, 2, 1), (3, 1, 1), (2, 2, 1)]   # the DSRF3D-lite network of Section 3
print("Receptive field of the last feature map of our DSRF3D-lite:", receptive_field(vgg_layers), "voxels =",
      receptive_field(vgg_layers) * VOXEL_A / 10, "nm")

🔍 **What just happened?** With the same 81 weights, three ordinary 3 × 3 × 3 layers see a 7-voxel (10.5 nm) cube —
less than half a ribosome — while dilations (1, 2, 4) see 15 voxels (22.5 nm). Notice the **checkerboard of holes** in
the dilated field: some voxels are skipped ("gridding"). Dense connectivity (each layer also receives the outputs of
*all* earlier layers) fills those holes, because the network can combine the views at every dilation. Pooling and
strides are the other way to grow the field: the last layer of the network we build next sees 21 voxels (31.5 nm),
enough to take in a whole ribosome.

### 📄 Paper connection — Dilated-DenseNet (Gao *et al.* 2020, 2021)

**Papers**: Gao S, Han R, Zeng X, Cui X, Liu Z, Xu M, Zhang F. *Dilated-DenseNet for macromolecule classification in
cryo-electron tomography.* ISBRA 2020, LNBI 12304, pp. 82–94, [doi:10.1007/978-3-030-57821-3_8](https://doi.org/10.1007/978-3-030-57821-3_8);
and Gao S, Han R, Zeng X, Liu Z, Xu M, Zhang F. *Macromolecules structural classification with a 3D dilated dense
network in cryo-electron tomography.* IEEE/ACM TCBB (2021), [doi:10.1109/TCBB.2021.3065986](https://doi.org/10.1109/TCBB.2021.3065986).

> ⚠️ We could not access the full text of these papers while writing; the description below is limited to the
> abstract. Read the papers before relying on details (dilation rates, growth rate, depth, input size and numbers are
> not given here on purpose).

- **Question it answers:** can a CNN classify noisy, heterogeneous, wedge-distorted subtomograms better than the CNNs of
  the SHREC contest?
- **What the paper does:** a 3D DenseNet whose dense blocks use **dilated** 3D convolutions, enlarging the receptive
  field without extra parameters or pooling; feature-map visualisations are used to argue that dilation gives more
  representative features.
- **What our toy keeps:** the receptive-field arithmetic and the "same weights, larger view" effect of dilation.
- **What our toy leaves out:** the dense blocks, the training, and the comparison with SHREC-CNN and 3D-DenseNet.
- **Result reported (abstract):** the authors report that 3D-Dilated-DenseNet significantly outperformed 3D-DenseNet,
  which in turn outperformed SHREC-CNN, on simulated (SHREC) and experimental data.
- **Code:** the AITom folder `aitom/classify/deep/supervised/cnn/dilated-densenet/` is an empty placeholder.

---
## 2 · A simulated subtomogram dataset

🧫 Real labelled subtomograms are scarce — an expert must decide what each box contains, and even experts disagree at
low SNR. That is why nearly every paper in this module trains on **simulated** subtomograms (Module 09): a known
molecule, a random orientation and position, the missing wedge of a ±60° tilt series, and noise.

We use the toolkit's fast *direct* simulator: rotate → Fourier wedge mask → white noise at a chosen SNR
(SNR = var(signal)/var(noise), Module 04). We simulate two noise levels: **SNR 0.1** (a good cellular tomogram) and
**SNR 0.03** (a hard one; the Xu-lab benchmarks go down to 0.01, where every network fails). The dataset is generated
once and **cached in memory**; every later section reuses it.

| split | per class | SNR | used for |
|---|---|---|---|
| train | 200 | half 0.1, half 0.03 | learning the weights |
| validation | 25 | mixed | watching training (never for final numbers) |
| test-0.1 / test-0.03 | 60 + 60 | 0.1 / 0.03 | final evaluation |
| unknown | 90 ClpB | 0.1 | Section 4 only — never used for training |

In [ ]:
# ── 2.1 Simulate the dataset once (cached in memory and reused everywhere) ──
def simulate_set(pids, n_per_class, snrs, rng, tilt_max_deg=60.0, max_shift=1.0):
    """Simulate n_per_class subtomograms of each molecule; the SNR cycles through `snrs`.

    Returns dict(X [N,1,24,24,24] normalised volumes, y [N] class index, snr [N],
    R [N,3,3] and t [N,3] = the true pose, kept only for the 'oracle alignment' of Section 4).
    """
    X, y, s, Rs, ts = [], [], [], [], []
    for k, pid in enumerate(pids):
        for i in range(n_per_class):
            snr = snrs[i % len(snrs)]
            v, R, t = simulate_subtomogram(TEMPLATES[pid], rng, snr=snr, tilt_max_deg=tilt_max_deg,
                                           max_shift=max_shift)
            X.append(normalize(v)); y.append(k); s.append(snr); Rs.append(R); ts.append(t)
    return dict(X=np.stack(X)[:, None], y=np.array(y), snr=np.array(s), R=np.stack(Rs), t=np.stack(ts))


t0 = time.time()
data_rng = np.random.default_rng(SEED)
TRAIN = simulate_set(CLASSES, 200, (0.1, 0.03), data_rng)
VAL = simulate_set(CLASSES, 25, (0.1, 0.03), data_rng)
TEST_HI = simulate_set(CLASSES, 60, (0.1,), data_rng)
TEST_LO = simulate_set(CLASSES, 60, (0.03,), data_rng)
UNK = simulate_set([UNKNOWN], 90, (0.1,), data_rng)
print(f"simulated {sum(len(d['y']) for d in [TRAIN, VAL, TEST_HI, TEST_LO, UNK])} subtomograms "
      f"in {time.time() - t0:.1f} s; train X: {TRAIN['X'].shape}, {TRAIN['X'].nbytes / 1e6:.0f} MB")

In [ ]:
# ── 2.2 Look at the data: clean template vs what the network receives ──
def side_view(t: np.ndarray) -> np.ndarray:
    """Projection along the axis that shows the largest silhouette (usually a side view)."""
    projs = [t.sum(a) for a in range(3)]
    return max(projs, key=lambda p: (p > 0.2 * p.max()).sum())


fig, axes = plt.subplots(len(CLASSES), 4, figsize=(6.4, 1.5 * len(CLASSES)))
for i, pid in enumerate(CLASSES):
    j_hi = np.where((TRAIN["y"] == i) & (TRAIN["snr"] == 0.1))[0][0]
    j_lo = np.where((TRAIN["y"] == i) & (TRAIN["snr"] == 0.03))[0][0]
    panels = [side_view(TEMPLATES[pid]), TRAIN["X"][j_hi, 0, BOX // 2], TRAIN["X"][j_hi, 0, :, BOX // 2],
              TRAIN["X"][j_lo, 0, BOX // 2]]
    for j, img in enumerate(panels):
        axes[i, j].imshow(img); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(SHORT[pid], fontsize=8)
for j, t in enumerate(["template\n(side projection)", "SNR 0.1\nxy slice", "SNR 0.1\nxz slice", "SNR 0.03\nxy slice"]):
    axes[0, j].set_title(t, fontsize=8)
fig.suptitle("Six classes as a 3D CNN sees them (24³ voxels, 36 nm box)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each row is one molecule. The clean projection (left) is what we would like to recognise; the
network only gets the noisy 3D box. The xz slice (third column) shows the **missing-wedge smearing along z** (Module 07).
At SNR 0.03 (right) most of us cannot tell the classes apart by eye — the network must pool evidence over all 13,824
voxels.

### 2.3 Augmentation that respects the microscope

**Data augmentation** (Module 05) shows the network modified copies of each training example so that it learns what
should *not* matter. For subtomograms the obvious choice is "rotate the molecule". But careful: every subtomogram
from a tomogram has its missing wedge in the **same** orientation (around the beam axis z). If we rotated a
subtomogram by an arbitrary angle, its wedge would rotate too, producing data no microscope can produce.

A neat compromise: a **180° rotation about the x, y or z axis** maps the wedge onto itself (the wedge mask is
symmetric under $k_i \to -k_i$). These are *proper* rotations (determinant +1). A single **mirror flip**, in contrast,
would preserve the wedge too but turn the molecule into its **mirror image** — biologically impossible, since proteins
are built from L-amino acids and α-helices are right-handed (chirality). Let us check the wedge claim numerically.

In [ ]:
# ── 2.3 Wedge-safe augmentation: 180° rotations keep the missing wedge, 90° rotations do not ──
def wedge_safe_augment(xb: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
    """Apply to each volume of a batch [B, 1, z, y, x] one of: identity or a 180° rotation about z, y or x.

    Flipping two axes at once = a 180° rotation about the third axis (a proper rotation, no mirror image).
    """
    out = xb.clone()
    choice = torch.randint(0, 4, (len(xb),), generator=gen)
    for c, dims in {1: (3, 4), 2: (2, 4), 3: (2, 3)}.items():   # about z, about y, about x
        sel = choice == c
        if sel.any():
            out[sel] = torch.flip(xb[sel], dims)
    return out


def power_outside_wedge(v: np.ndarray, mask: np.ndarray) -> float:
    """Fraction of Fourier power (excluding DC) that falls inside the missing (unmeasured) region."""
    P = np.abs(np.fft.fftn(v)) ** 2
    P[0, 0, 0] = 0
    return float(P[mask == 0].sum() / P.sum())


wedge = missing_wedge_mask((BOX,) * 3, 60)
# a noise-free, wedge-filtered capped proteasome, so that the empty wedge is easy to see
vol = apply_fourier_mask(rotate_volume(TEMPLATES["6utj"], random_rotation(np.random.default_rng(3))), wedge)
print(f"wedge-filtered subtomogram       : {100 * power_outside_wedge(vol, wedge):5.1f} % of power inside the missing wedge")
for name, dims in [("180° about z", (1, 2)), ("180° about y", (0, 2)), ("180° about x", (0, 1))]:
    print(f"after {name:27s}: {100 * power_outside_wedge(np.flip(vol, dims), wedge):5.1f} %")
rot90 = np.rot90(vol, axes=(0, 1))
print(f"after 90° about x (z↔y swapped)  : {100 * power_outside_wedge(rot90, wedge):5.1f} %  ← an impossible tomogram")
assert power_outside_wedge(np.flip(vol, (1, 2)), wedge) < 1e-3, "180° rotations should keep the wedge empty"
assert power_outside_wedge(rot90, wedge) > 0.05, "a 90° rotation about x should move signal into the wedge"

🔍 **What just happened?** After any of the three 180° rotations the wedge region stays empty (0 % of the power),
whereas a 90° rotation about x moves the wedge to a direction where it never is in a real tomogram, so signal appears
where the microscope could not have measured anything. Our training function applies the safe rotations to every
batch. (In our fast simulator the white noise is added *after* the wedge, so noisy subtomograms do have noise power
inside the wedge; in a real reconstruction the noise is wedge-shaped too — one of the many small differences between
simulation and reality that Module 20 is about.)

---
## 3 · Two architectures: DSRF3D-lite and RB3D-lite

The Xu lab's first subtomogram CNNs (2017–2018) borrowed two ideas from 2D image recognition:

- **VGG-style stacks** of small 3 × 3 × 3 convolutions with max-pooling between them (**DSRF3D**, "deep small receptive
  field"; Xu *et al.* 2017), later made deeper (**DSRF3D-v2**; Che *et al.* 2018). Two 3³ layers see as far as one 5³
  layer but with fewer weights and an extra nonlinearity.
- **Residual blocks** (**RB3D**; Che *et al.* 2018): a block computes $y = \mathrm{ReLU}(F(x) + S(x))$, where $F$ is a
  small stack of convolutions (1 × 1 × 1 → 3 × 3 × 3 → 1 × 1 × 1, a "bottleneck") and $S$ is a shortcut. *In words:*
  the block only has to learn a **correction** $F$ to what the shortcut already passes on, which makes deep networks
  much easier to train.

| network (paper) | layers in the paper | our "lite" version (tens of thousands of weights) |
|---|---|---|
| DSRF3D (Xu 2017) | 4 conv 3³ (32-32-64-64), 2 max-pools, FC-512, FC-512 | stride-2 stem, 2 conv, 2 pools, one FC layer |
| DSRF3D-v2 (Che 2018) | 6 conv (32-32-64-64-128-128), 3 pools, 2 FC | (not built) |
| RB3D (Che 2018) | conv + pool, 4 bottleneck residual blocks (3³ conv shortcut), 2 FC (50 % dropout) | stem, 2 bottleneck blocks, one FC layer |
| CB3D (Che 2018) | C3D-inspired: 8 conv, 5 pools, first pool 1 × 2 × 2 | (not built) |

Three departures, all for CPU speed: our first convolution has **stride 2** (it halves the box to 12³ at once); we
add **batch normalisation**, a later standard trick that lets training converge in a few epochs; and we use a single
fully connected layer (no dropout), which converged fastest in our tests.

In [ ]:
# ── 3.1 Build the two networks ──
def conv_bn_relu(c_in, c_out, stride=1):
    """3×3×3 convolution → batch normalisation → ReLU (the basic VGG-style unit)."""
    return [nn.Conv3d(c_in, c_out, 3, stride=stride, padding=1), nn.BatchNorm3d(c_out), nn.ReLU()]


class DSRF3DLite(nn.Module):
    """A small VGG-style 3D CNN in the spirit of DSRF3D (Xu et al. 2017; Che et al. 2018)."""

    def __init__(self, n_classes=6, w=8):
        super().__init__()
        self.features = nn.Sequential(
            *conv_bn_relu(1, w, stride=2),                     # 24³ → 12³
            *conv_bn_relu(w, 2 * w), nn.MaxPool3d(2),          # 12³ → 6³
            *conv_bn_relu(2 * w, 4 * w), nn.MaxPool3d(2))      # 6³ → 3³
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(4 * w * 27, n_classes))   # pre-softmax outputs f_j(x)

    def forward(self, x):
        return self.head(self.features(x))


class Bottleneck3D(nn.Module):
    """Residual bottleneck: 1×1×1 → 3×3×3 → 1×1×1 main path plus a 3×3×3 convolution shortcut (as described for RB3D)."""

    def __init__(self, c_in, c_out):
        super().__init__()
        mid = max(c_out // 4, 2)
        self.main = nn.Sequential(nn.Conv3d(c_in, mid, 1), nn.BatchNorm3d(mid), nn.ReLU(),
                                  nn.Conv3d(mid, mid, 3, padding=1), nn.BatchNorm3d(mid), nn.ReLU(),
                                  nn.Conv3d(mid, c_out, 1), nn.BatchNorm3d(c_out))
        self.shortcut = nn.Sequential(nn.Conv3d(c_in, c_out, 3, padding=1), nn.BatchNorm3d(c_out))

    def forward(self, x):
        return F.relu(self.main(x) + self.shortcut(x))


class RB3DLite(nn.Module):
    """A small residual 3D CNN in the spirit of RB3D (Che et al. 2018)."""

    def __init__(self, n_classes=6, w=8):
        super().__init__()
        self.features = nn.Sequential(*conv_bn_relu(1, w, stride=2),
                                      Bottleneck3D(w, 2 * w), nn.MaxPool3d(2),
                                      Bottleneck3D(2 * w, 4 * w), nn.MaxPool3d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(4 * w * 27, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def n_params(model) -> int:
    """Number of trainable weights."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for name, m in [("DSRF3D-lite", DSRF3DLite()), ("RB3D-lite", RB3DLite())]:
    out = m(torch.zeros(2, 1, BOX, BOX, BOX))
    print(f"{name:12s}: {n_params(m):7,d} weights; input [2, 1, 24, 24, 24] → output {list(out.shape)}")

### 3.2 One training function for the whole notebook

Every experiment in this notebook (plain training, class re-weighting, focal loss, mixup, distillation, fine-tuning)
is the same loop with a different **loss**. We therefore write the loss in its most general form, for a *target
distribution* $t$ over classes (a one-hot vector for ordinary labels; a soft vector for mixup and distillation) and the
network's predicted probabilities $p = \mathrm{softmax}(z/T)$ from its pre-softmax outputs $z$:

$$\mathcal L \;=\; -\,T^2 \sum_{c} w_c\; t_c\; (1 - p_c)^{\gamma}\; \log p_c .$$

*In words:* the usual cross-entropy ($w_c = 1$, $\gamma = 0$, $T = 1$) punishes a low probability for the correct class.
The extra knobs — class weights $w_c$, the focal exponent $\gamma$, the temperature $T$ — appear in Sections 5–6. We use
the Adam optimizer whose learning rate decays smoothly to zero (a cosine schedule); the papers used SGD with Nesterov
momentum, but Adam converges faster in a few CPU epochs.

In [ ]:
# ── 3.2 Training and prediction helpers ──
def one_hot(y, n=len(CLASSES)) -> np.ndarray:
    """Integer labels → one-hot target distributions."""
    return np.eye(n, dtype=np.float32)[y]


@torch.no_grad()
def predict_logits(model, X, batch=128) -> np.ndarray:
    """Pre-softmax outputs z = f(x) for every volume (model in evaluation mode)."""
    model.eval()
    out = [model(torch.as_tensor(X[i:i + batch]).to(DEVICE)).cpu() for i in range(0, len(X), batch)]
    return torch.cat(out).numpy()


def accuracy(model, X, y) -> float:
    """Fraction of volumes whose highest-scoring class is the true class."""
    return float((predict_logits(model, X).argmax(1) == y).mean())


def train_classifier(model, X, targets, epochs=6, batch=32, lr=2e-3, gamma=0.0, class_weights=None, T=1.0,
                     mixup_alpha=0.0, val=None, seed=SEED, verbose=True):
    """Train `model` on volumes X with target distributions `targets` [N, C] (one-hot or soft).

    Loss = −T² Σ_c w_c t_c (1 − p_c)^γ log p_c with p = softmax(z / T)  (γ = 0, w = 1, T = 1 → cross-entropy).
    mixup_alpha > 0 → train on λ·x_i + (1 − λ)·x_j with targets λ·t_i + (1 − λ)·t_j, λ ~ Beta(α, α).
    Every batch gets wedge-safe 180° rotations. Returns a history dict (loss, val_acc, seconds).
    """
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    mix_rng = np.random.default_rng(seed)
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * math.ceil(len(X) / batch))
    Xt, Tt = torch.as_tensor(X), torch.as_tensor(targets, dtype=torch.float32)
    w = None if class_weights is None else torch.as_tensor(class_weights, dtype=torch.float32, device=DEVICE)
    hist, t0 = dict(loss=[], val_acc=[]), time.time()
    for ep in range(epochs):
        model.train()
        perm, total = torch.randperm(len(X), generator=gen), 0.0
        for i in range(0, len(X), batch):
            idx = perm[i:i + batch]
            xb, tb = wedge_safe_augment(Xt[idx], gen).to(DEVICE), Tt[idx].to(DEVICE)
            if mixup_alpha > 0:
                lam = float(mix_rng.beta(mixup_alpha, mixup_alpha))
                j = torch.randperm(len(idx), generator=gen).to(DEVICE)
                xb, tb = lam * xb + (1 - lam) * xb[j], lam * tb + (1 - lam) * tb[j]
            logp = F.log_softmax(model(xb) / T, dim=1)
            per_class = -tb * logp * ((1 - logp.exp()) ** gamma if gamma > 0 else 1.0)
            if w is not None:
                per_class = per_class * w
            loss = per_class.sum(1).mean() * T * T
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(idx)
        hist["loss"].append(total / len(X))
        if val is not None:
            hist["val_acc"].append(accuracy(model, *val))
        if verbose:
            v = f"  val acc {hist['val_acc'][-1]:.2f}" if val is not None else ""
            print(f"  epoch {ep + 1:2d}/{epochs}  loss {hist['loss'][-1]:.3f}{v}  ({time.time() - t0:.0f} s)")
    hist["seconds"] = time.time() - t0
    return hist


print("helpers ready: one_hot, predict_logits, accuracy, train_classifier")

### 3.3 Train both networks

Same data, same number of epochs, same seed: only the architecture differs. (On a free Colab CPU each training takes
roughly 20–90 s depending on the CPU; the timing is printed.)

In [ ]:
# ── 3.3a Train DSRF3D-lite ──
torch.manual_seed(SEED)
dsrf = DSRF3DLite()
print(f"DSRF3D-lite ({n_params(dsrf):,} weights)")
hist_dsrf = train_classifier(dsrf, TRAIN["X"], one_hot(TRAIN["y"]), epochs=7, val=(VAL["X"], VAL["y"]))

In [ ]:
# ── 3.3b Train RB3D-lite ──
torch.manual_seed(SEED)
rb3d = RB3DLite()
print(f"RB3D-lite ({n_params(rb3d):,} weights)")
hist_rb = train_classifier(rb3d, TRAIN["X"], one_hot(TRAIN["y"]), epochs=7, val=(VAL["X"], VAL["y"]))

In [ ]:
# ── 3.4 Compare: learning curves and test accuracy at two noise levels ──
results = {}
for name, m, h in [("DSRF3D-lite", dsrf, hist_dsrf), ("RB3D-lite", rb3d, hist_rb)]:
    results[name] = dict(hi=accuracy(m, TEST_HI["X"], TEST_HI["y"]), lo=accuracy(m, TEST_LO["X"], TEST_LO["y"]),
                         params=n_params(m), sec=h["seconds"])
    print(f"{name:12s} test accuracy: SNR 0.1 → {results[name]['hi']:.2f} | SNR 0.03 → {results[name]['lo']:.2f}"
          f"   ({results[name]['params']:,} weights, trained in {h['seconds']:.0f} s; chance = {1 / len(CLASSES):.2f})")

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3))
for name, h, c in [("DSRF3D-lite", hist_dsrf, "tab:blue"), ("RB3D-lite", hist_rb, "tab:orange")]:
    ep = np.arange(1, len(h["loss"]) + 1)
    axes[0].plot(ep, h["loss"], "o-", color=c, label=name)
    axes[1].plot(ep, h["val_acc"], "o-", color=c, label=name)
axes[0].set(xlabel="epoch", ylabel="training loss", title="Training loss")
axes[1].set(xlabel="epoch", ylabel="validation accuracy", title="Validation accuracy (mixed SNR)", ylim=(0, 1))
axes[1].axhline(1 / len(CLASSES), color="gray", ls=":", label="chance")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.5 Where do the errors go? Confusion matrices ──
def plot_confusion(ax, y_true, y_pred, names, title):
    """Row-normalised confusion matrix (rows = true class, columns = predicted)."""
    cm = confusion_matrix(y_true, y_pred, labels=range(len(names))).astype(float)
    cm /= cm.sum(1, keepdims=True)
    ax.imshow(cm, cmap="viridis", vmin=0, vmax=1)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="black" if cm[i, j] > 0.5 else "white")
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=9)


best_name = max(results, key=lambda k: results[k]["lo"])
best = dsrf if best_name == "DSRF3D-lite" else rb3d
names = [SHORT[p] for p in CLASSES]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (lab, d) in zip(axes, [("SNR 0.1", TEST_HI), ("SNR 0.03", TEST_LO)]):
    plot_confusion(ax, d["y"], predict_logits(best, d["X"]).argmax(1), names, f"{best_name}, test {lab}")
plt.tight_layout(); plt.show()
assert results[best_name]["hi"] > 0.5, "the classifier should beat chance (0.17) by far at SNR 0.1"

🔍 **What just happened?** Both networks learned to classify six molecules from noisy, wedge-distorted boxes far above
chance (1/6 ≈ 0.17), and both lose accuracy at SNR 0.03 — the same downward trend the papers report (and, below
about SNR 0.01, every published network collapses to chance). Read the confusion matrices row by row: most errors
stay **inside a biological family** — 70S ↔ 80S ribosome, and among the three proteasome assemblies, which share
the same 20S core and differ only by their caps. RuBisCO, small and compact, is easy to tell apart from everything
else. In our short training the residual RB3D-lite learns more slowly than DSRF3D-lite (look at its validation curve:
still rising after seven epochs) and costs about twice the time per epoch on a CPU. Do not conclude that residual
networks are worse: in the paper, the deeper RB3D and CB3D beat DSRF3D in most settings after full training — residual
connections pay off when networks are *deep* and trained long, which our CPU budget does not allow.

✏️ **Your turn.** Change `epochs=7` to `epochs=12` in cell 3.3a (or `w=8` to `w=12` in `DSRF3DLite`) and re-run 3.3a
and 3.4. Does accuracy at SNR 0.03 improve more than at SNR 0.1? Why might more capacity help most when the signal is
weakest?

### 📄 Paper connection — improved 3D CNNs (Che *et al.* 2018)

**Paper**: Che C, Lin R, Zeng X, Elmaaroufi K, Galeotti J, Xu M. *Improved deep learning-based macromolecules structure
classification from electron cryo-tomograms.* Machine Vision and Applications 29(8):1227–1236 (2018).
[doi:10.1007/s00138-018-0949-4](https://doi.org/10.1007/s00138-018-0949-4) · [arXiv:1707.04885](https://arxiv.org/abs/1707.04885)

- **Question it answers:** can deeper or differently designed 3D CNNs beat DSRF3D, especially at low SNR and with a
  large missing wedge?
- **What the paper actually does:** proposes **DSRF3D-v2** (VGG-style, 6 convolutions + 3 pools, 2 FC layers),
  **RB3D** (3D residual network: conv + pool, four bottleneck blocks with a 1 × 1 → 3 × 3 → 1 × 1 main path and a
  3 × 3 convolution shortcut, two FC layers with 50 % dropout) and **CB3D** (C3D-inspired video-style network: 8 convs,
  5 pools, the first pool 1 × 2 × 2). Trained with cross-entropy, SGD + Nesterov momentum (lr 0.005), batch 64,
  ≤ 20 epochs with early stopping, on the lab's simulated benchmark (22 complexes + an empty "NULL" class; 40³ voxels;
  SNR 0.1–0.01; tilt ±60/50/40°).
- **What our toy keeps:** VGG-style and residual-bottleneck designs, wedge + noise simulation, accuracy vs SNR.
- **What our toy leaves out:** depth (we have ≈ 4–6 layers), CB3D, the 23-class benchmark, early stopping, SNR 0.01.
- **Result reported:** the new models beat DSRF3D in 11 of 12 SNR/tilt settings, CB3D was best in 8 of 12; at ±60°,
  SNR 0.1: CB3D 0.973 vs DSRF3D 0.911; SNR 0.03: 0.849 vs 0.706; at SNR 0.01 almost all models fall to ≈ 0.04
  (chance for 23 classes).
- **Code:** no dedicated AITom folder. PyTorch re-implementations of CB3D and DSRF3D-v2 live in
  `aitom/classify/deep/unsupervised/cryo_shift/models/` (written for the later Cryo-Shift paper); ⚠️ that CB3D has 6
  convolutions and 4 pools, so it is **not** the paper's exact architecture.

---
## 4 · Subdivision: CNN features → k-means → averages (Xu *et al.* 2017)

🧫 **The scaling problem.** Template-free classification and averaging (Modules 14–15) needs all-against-all alignment:
for $N$ subtomograms that is $\sim N^2$ alignments of seconds each — impossible for $N = 10^5$. A supervised CNN is
fast but only knows its training classes. Xu *et al.* (2017) combined the two strengths:

1. Train a CNN classifier on simulated subtomograms of **known** structures.
2. **Remove the softmax.** Use the $L$ pre-softmax outputs as a feature vector:

   $$P(j \mid \mathbf x) = \frac{e^{f_j(\mathbf x)}}{\sum_{l=1}^{L} e^{f_l(\mathbf x)}}, \qquad
   \text{feature}_j(\mathbf x) = f_j(\mathbf x), \quad j = 1, \dots, L.$$

   *In words:* $f_j(\mathbf x)$ measures "how much does this subtomogram look like training structure $j$". Even a
   molecule outside the training set gets a characteristic *profile* of similarities — the paper calls this a
   nonlinear projection onto the space spanned by the training structures.
3. Cluster these features with **k-means using many more clusters than expected classes** (k = 100 in the paper) —
   deliberate **over-partitioning**, so each cluster is small and nearly pure.
4. Run careful reference-free alignment and averaging **inside each cluster** (divide and conquer).

Our test: a "cell" containing our six known molecules **plus 90 copies of ClpB**, a molecule the network has never seen.

In [ ]:
# ── 4.1 Features = pre-softmax outputs of the trained CNN; compare with raw voxels ──
MIX = dict(X=np.concatenate([TEST_HI["X"], UNK["X"]]), y=np.concatenate([TEST_HI["y"], UNK["y"] + len(CLASSES)]),
           R=np.concatenate([TEST_HI["R"], UNK["R"]]), t=np.concatenate([TEST_HI["t"], UNK["t"]]))
MIX_NAMES = [SHORT[p] for p in CLASSES] + [SHORT[UNKNOWN] + " (unseen)"]
K_SUB = 14                                   # over-partition: 2 × the 7 molecules actually present

feats = predict_logits(best, MIX["X"])                                       # [N, 6] "structural features"
raw_pca = PCA(n_components=20, random_state=SEED).fit_transform(MIX["X"].reshape(len(MIX["X"]), -1))


def purity(y_true, clusters) -> float:
    """Fraction of items that belong to the majority true class of their cluster."""
    return sum(np.bincount(y_true[clusters == c]).max() for c in np.unique(clusters)) / len(y_true)


sub = {}
for name, Z in [("CNN pre-softmax features", feats), ("raw voxels (PCA-20)", raw_pca)]:
    cl = KMeans(n_clusters=K_SUB, n_init=10, random_state=SEED).fit_predict(Z)
    sub[name] = cl
    print(f"{name:26s}: k = {K_SUB}  purity {purity(MIX['y'], cl):.2f}   ARI {adjusted_rand_score(MIX['y'], cl):.2f}")

fig, ax = plt.subplots(figsize=(5, 4))
Z2 = PCA(n_components=2, random_state=SEED).fit_transform(feats)
for k, nm in enumerate(MIX_NAMES):
    sel = MIX["y"] == k
    ax.scatter(*Z2[sel].T, s=8, label=nm, color="black" if k == len(CLASSES) else plt.cm.tab10(k),
               marker="x" if k == len(CLASSES) else "o")
ax.set_xlabel("PC 1 of the 6 pre-softmax outputs"); ax.set_ylabel("PC 2")
ax.set_title("Subtomograms in the CNN's feature space (SNR 0.1)")
ax.legend(fontsize=7, markerscale=2, loc="best")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In raw-voxel space, two copies of the same molecule in different orientations look very
different, so clustering raw voxels mostly groups by noise and pose (low purity). In the CNN's feature space the
molecules separate — including ClpB (black crosses): although no output unit was ever trained to say "ClpB", its
*pattern* of similarities to the six known classes is distinctive enough to form its own region. (Adjusted Rand index,
ARI, Module 04: 0 = random grouping, 1 = perfect.)

In [ ]:
# ── 4.2 What did each cluster capture? ──
cl = sub["CNN pre-softmax features"]
table = np.zeros((len(MIX_NAMES), K_SUB), int)
np.add.at(table, (MIX["y"], cl), 1)
order = np.lexsort((-table.max(0), table.argmax(0)))                 # sort clusters by their majority class
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.imshow(table[:, order], cmap="viridis")
for i in range(table.shape[0]):
    for j, c in enumerate(order):
        if table[i, c]:
            ax.text(j, i, table[i, c], ha="center", va="center", fontsize=7, color="white" if table[i, c] < 30 else "k")
ax.set_yticks(range(len(MIX_NAMES))); ax.set_yticklabels(MIX_NAMES, fontsize=8)
ax.set_xticks(range(K_SUB)); ax.set_xticklabels(order, fontsize=7); ax.set_xlabel("cluster id")
ax.set_title(f"Subdivision: {len(MIX['y'])} subtomograms → {K_SUB} clusters (counts)")
plt.tight_layout(); plt.show()

### 4.3 Averaging inside each cluster

Each cluster is now small and fairly homogeneous, so we can afford to align and average it. Alignment is the subject of
Modules 12–14; here, to keep the focus on subdivision, we use **oracle alignment**: the simulator recorded every
particle's true rotation $R$ and shift $\mathbf t$, so we simply undo them (`rotate_volume(v, Rᵀ, −Rᵀt)`). In real data
this step is replaced by reference-free alignment (Module 14), which the paper ran with the TomoMiner software. Because
every copy had its missing wedge in a different orientation *relative to the molecule*, the average fills in the
wedge. We then ask: which template does each cluster average resemble most?

In [ ]:
# ── 4.3 Oracle-aligned cluster averages and their best-matching template ──
def oracle_average(X, R, t) -> np.ndarray:
    """Average subtomograms after undoing each one's known pose (stand-in for real alignment, Module 14)."""
    acc = np.zeros(X.shape[2:], np.float32)
    for v, Ri, ti in zip(X[:, 0], R, t):
        acc += rotate_volume(v, Ri.T, -Ri.T @ ti)
    return acc / len(X)


refs = {SHORT[p]: TEMPLATES[p] for p in CLASSES + [UNKNOWN]}
rows = []
for c in order:
    members = np.where(cl == c)[0]
    if len(members) < 5:
        continue
    avg = oracle_average(MIX["X"][members], MIX["R"][members], MIX["t"][members])
    scores = {nm: pearson(avg, ref) for nm, ref in refs.items()}
    match = max(scores, key=scores.get)
    majority = MIX_NAMES[np.bincount(MIX["y"][members], minlength=len(MIX_NAMES)).argmax()]
    rows.append((c, len(members), majority, match, scores[match], avg))
    print(f"cluster {c:2d}: {len(members):3d} members, majority = {majority:17s} → average best matches "
          f"{match:12s} (r = {scores[match]:.2f})")

clpb_rows = [r for r in rows if r[2].startswith("ClpB")]
largest = {}                                                   # the largest cluster of each majority molecule
for row in rows:
    if row[2] not in largest or row[1] > largest[row[2]][1]:
        largest[row[2]] = row
show = [largest[nm] for nm in MIX_NAMES if nm in largest]
fig, axes = plt.subplots(2, len(show), figsize=(1.3 * len(show), 3.6))
for j, (c, n, maj, match, r, avg) in enumerate(show):
    axis = int(np.argmax([(refs[match].sum(a) > 0.2 * refs[match].sum(a).max()).sum() for a in range(3)]))
    axes[0, j].imshow(avg.sum(axis)); axes[0, j].set_title(f"cluster {c}, n={n}\nmostly {maj.split(' (')[0]}", fontsize=7)
    axes[1, j].imshow(refs[match].sum(axis)); axes[1, j].set_title(f"best match:\n{match} (r={r:.2f})", fontsize=7)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Top: oracle-aligned cluster averages (side projections). Bottom: best-matching template", fontsize=9)
plt.tight_layout(); plt.show()
if clpb_rows:
    print(f"ClpB-dominated cluster(s): {[int(r[0]) for r in clpb_rows]} → best match {[r[3] for r in clpb_rows]}")

🔍 **What just happened?** The CNN never saw ClpB, yet the subdivision put most ClpB copies into cluster(s) of their
own, and their aligned average looks like — and correlates best with — the ClpB template. This is the leave-one-out
experiment of Xu *et al.* (2017) in miniature: a **supervised** network trained on known molecules became a fast,
general-purpose **pre-sorting** tool for template-free discovery. Mixed clusters (if you see any) are exactly why the
paper over-partitions and then relies on careful alignment-based classification inside each cluster.

⚠️ Our averages used the *true* poses. In the paper, each cluster went through reference-free subtomogram alignment and
classification, which is slower and can fail for tiny or very mixed clusters.

✏️ **Your turn.** Set `K_SUB = 7` (exactly the number of molecules) and re-run 4.1–4.3. Do clusters become less pure?
Then try `K_SUB = 25`. What is the trade-off between purity and the number of copies available to each average?

### 📄 Paper connection — deep-learning subdivision (Xu *et al.* 2017)

**Paper**: Xu M, Chai X, Muthakana H, Liang X, Yang G, Zeev-Ben-Mordehai T, Xing E. *Deep learning based subdivision
approach for large scale macromolecules structure recovery from electron cryo tomograms.* Bioinformatics
33(14):i13–i22 (ISMB/ECCB 2017). [doi:10.1093/bioinformatics/btx230](https://doi.org/10.1093/bioinformatics/btx230) ·
[arXiv:1701.08404](https://arxiv.org/abs/1701.08404)

- **Question it answers:** how can structure recovery scale to $10^4$–$10^6$ heterogeneous subtomograms?
- **What the paper actually does:** trains two 3D CNNs — **Inception3D** (one 3D inception module: parallel 1³, 3³, 5³
  convolutions and a pooling branch) and **DSRF3D** (VGG-like: 32-32-pool-64-64-pool, FC-512, FC-512) — on simulated
  40³ subtomograms of 22 complexes plus an empty class; removes the softmax and uses the pre-softmax outputs as
  structural features; **k-means with k = 100** over-partitions the data; reference-free (TomoMiner-based) averaging
  then runs inside each cluster.
- **What our toy keeps:** pre-softmax features, deliberate over-partitioning, per-cluster averaging, an unseen class.
- **What our toy leaves out:** Inception3D; 23 classes at 40³ and 0.92 nm; real alignment (we used oracle poses);
  FSC-based evaluation of the averages; the experimental data.
- **Result reported:** at SNR 0.03, ±60°, the authors report accuracies of 0.608 (Inception3D) and 0.658 (DSRF3D) vs
  0.446 for a rotation-invariant-feature SVM; on experimental data (purified *E. coli* ribosome and human 20S proteasome,
  401 subtomograms) Inception3D reached 0.905, while DSRF3D did not converge on this small set; in leave-one-class-out
  tests the held-out structure was recovered to better than 7 nm for 16/22 (Inception3D) and 18/22 (DSRF3D) classes;
  feature extraction took ≈ 0.0015 s per subtomogram.
- **Code:** AITom `aitom/classify/deep/supervised/cnn/subdivide.py` (Keras; note `Dropout(0.7)` is a *drop* rate).

---
## 5 · Rare molecules: learning from imbalanced classes (Luo *et al.* 2019)

🧫 Cells are not balanced datasets. In a neuron, proteasomes may outnumber TRiC chaperonins ten to one; in a bacterium,
ribosomes vastly outnumber almost everything. In the rat-neuron data used by Luo *et al.* (2019) the four classes had
1,043, 386, 125 and 80 subtomograms. A network trained naively on such data learns that "when in doubt, say the common
class" — and **overall accuracy hides the failure**, because the rare classes contribute few test examples.

**Better metrics.** Per-class **recall** (sensitivity) $= TP_c / (TP_c + FN_c)$; **macro-F1** = the unweighted mean of
per-class F1 scores; **G-mean** = the geometric mean of per-class recalls, $\big(\prod_c \text{recall}_c\big)^{1/C}$,
which is 0 if *any* class is never found.

**Three remedies**, all expressible with our general loss:

| remedy | formula | idea |
|---|---|---|
| class re-weighting | $w_c \propto 1/n_c$ | each class contributes equally to the loss |
| focal loss (Lin *et al.* 2017) | $\mathrm{FL}(p_t) = -\alpha_t (1-p_t)^{\gamma}\log p_t$, $\gamma = 2$ | easy, confident examples (mostly the common class) are down-weighted |
| mixup (Zhang *et al.* 2018) | $\tilde x = \lambda x_i + (1-\lambda)x_j$, $\tilde y = \lambda y_i + (1-\lambda)y_j$ | train on blends of examples: smoother decision boundaries, a regulariser |

**The setting.** Imagine you have a feature extractor trained on balanced *simulated* data — the convolutional layers
of our DSRF3D-lite from Section 3 — and must now train a classifier for an *imbalanced* experimental dataset. We
freeze the convolutional layers and train only a **new classification head** on a long-tailed subset of the cached
training data (150 → 10 copies per class), four times with four losses. Freezing keeps each training to seconds on a
CPU. (Luo *et al.* trained their whole multi-path network on the imbalanced data.) We test on the **balanced** test
sets (both SNRs, 120 copies per class).

In [ ]:
# ── 5.1 A long-tailed training set and frozen features ──
LONG_TAIL = {"4v4r": 150, "3j9i": 80, "1bxn": 45, "6utj": 25, "4cr2": 15, "4v7r": 10}
idx_lt = np.concatenate([np.where(TRAIN["y"] == CLASSES.index(p))[0][:n] for p, n in LONG_TAIL.items()])
X_lt, y_lt = TRAIN["X"][idx_lt], TRAIN["y"][idx_lt]
counts = np.bincount(y_lt, minlength=len(CLASSES))
print("training counts per class:", {nm: int(c) for nm, c in zip(names, counts)},
      f"(total {len(y_lt)}, ratio {counts.max() // counts.min()}:1)")


@torch.no_grad()
def frozen_features(X, batch=128) -> torch.Tensor:
    """Flattened output of the frozen convolutional layers of the Section-3 classifier."""
    best.eval()
    return torch.cat([best.features(torch.as_tensor(X[i:i + batch]).to(DEVICE)).flatten(1).cpu()
                      for i in range(0, len(X), batch)])


X_bal = np.concatenate([TEST_HI["X"], TEST_LO["X"]]); y_bal = np.concatenate([TEST_HI["y"], TEST_LO["y"]])
F_lt, F_bal = frozen_features(X_lt), frozen_features(X_bal)
# mixup needs features of *blended volumes*: pre-compute 4 random blends per training example
mix_rng = np.random.default_rng(SEED)
pairs = mix_rng.integers(0, len(X_lt), size=(4 * len(X_lt), 2))
lam = mix_rng.beta(0.4, 0.4, size=len(pairs)).astype(np.float32)
X_mix = lam[:, None, None, None, None] * X_lt[pairs[:, 0]] + (1 - lam[:, None, None, None, None]) * X_lt[pairs[:, 1]]
F_mix = frozen_features(X_mix)
T_mix = lam[:, None] * one_hot(y_lt[pairs[:, 0]]) + (1 - lam[:, None]) * one_hot(y_lt[pairs[:, 1]])
print(f"features: long-tail {tuple(F_lt.shape)}, balanced test {tuple(F_bal.shape)}, mixup blends {tuple(F_mix.shape)}")

In [ ]:
# ── 5.2 Four ways to train a new head on the same long-tailed data ──
def train_head(F_tr, T_tr, gamma=0.0, class_weights=None, epochs=60, lr=3e-3, seed=SEED):
    """Train a fresh linear head on frozen features with the general loss of Section 3.2."""
    torch.manual_seed(seed)
    head = nn.Linear(F_tr.shape[1], len(CLASSES))
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    Tt = torch.as_tensor(T_tr, dtype=torch.float32)
    w = None if class_weights is None else torch.as_tensor(class_weights, dtype=torch.float32)
    for _ in range(epochs):
        for idx in torch.randperm(len(F_tr)).split(32):
            logp = F.log_softmax(head(F_tr[idx]), 1)
            per_class = -Tt[idx] * logp * ((1 - logp.exp()) ** gamma if gamma > 0 else 1.0)
            loss = (per_class * (w if w is not None else 1.0)).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return head


def imbalance_report(head, name):
    """Per-class recall, macro-F1 and G-mean on the balanced test sets."""
    with torch.no_grad():
        pred = head(F_bal).argmax(1).numpy()
    rec = np.array([(pred[y_bal == c] == c).mean() for c in range(len(CLASSES))])
    out = dict(recall=rec, acc=float((pred == y_bal).mean()),
               macro_f1=f1_score(y_bal, pred, average="macro"), gmean=float(np.prod(rec) ** (1 / len(rec))))
    print(f"{name:16s} acc {out['acc']:.2f}  macro-F1 {out['macro_f1']:.2f}  G-mean {out['gmean']:.2f}  "
          f"recall of the 3 rarest (capped, 26S, 80S): {np.round(rec[[CLASSES.index(p) for p in ['6utj', '4cr2', '4v7r']]], 2)}")
    return out


inv_freq = counts.sum() / (len(CLASSES) * counts)                    # w_c ∝ 1 / n_c (mean weight ≈ 1)
t0 = time.time()
imb = {"cross-entropy": imbalance_report(train_head(F_lt, one_hot(y_lt)), "cross-entropy"),
       "re-weighted CE": imbalance_report(train_head(F_lt, one_hot(y_lt), class_weights=inv_freq), "re-weighted CE"),
       "focal (γ = 2)": imbalance_report(train_head(F_lt, one_hot(y_lt), gamma=2.0), "focal (γ = 2)"),
       "mixup + focal": imbalance_report(train_head(torch.cat([F_lt, F_mix]),
                                                    np.concatenate([one_hot(y_lt), T_mix]), gamma=2.0), "mixup + focal")}
print(f"(4 head trainings in {time.time() - t0:.0f} s)")

order_lt = np.argsort(-counts)                                       # plot classes from common to rare
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), gridspec_kw=dict(width_ratios=[1, 2.2]))
axes[0].bar([names[c] for c in order_lt], counts[order_lt], color="gray"); axes[0].set_ylabel("training examples")
axes[0].set_title("Long-tailed training set"); axes[0].tick_params(axis="x", rotation=45, labelsize=7)
wbar = 0.2
for i, (name, r) in enumerate(imb.items()):
    axes[1].bar(np.arange(len(CLASSES)) + (i - 1.5) * wbar, r["recall"][order_lt], wbar,
                label=f"{name} (macro-F1 {r['macro_f1']:.2f}, G-mean {r['gmean']:.2f})")
axes[1].set_xticks(range(len(CLASSES))); axes[1].set_xticklabels([names[c] for c in order_lt], rotation=45, fontsize=7)
axes[1].set_ylabel("recall on balanced test sets"); axes[1].set_ylim(0, 1.05)
axes[1].set_title("Per-class recall, classes ordered from common (left) to rare (right)")
axes[1].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The frozen features are the same in all four runs; only the head and its loss change. With
plain cross-entropy the head learns the training **prior** — when in doubt, choose the common class — so the rare
classes (right side of the chart) lose recall, and the G-mean is low. Re-weighting moves the decision boundaries back
towards the common classes and buys rare-class recall (here for the capped and 26S proteasomes) at little cost. The focal
loss changes little, and mixup + focal even hurts in this setting: with a *frozen* feature extractor, a blended volume
does not land "between" its two parents in feature space, so the soft targets mislead the head (Luo *et al.* applied
mixup while training the whole network). One rare class is lost by every method: the 80S ribosome, with only 10
training copies and an abundant look-alike, the 70S ribosome. Which remedy wins depends on the data and the seed, so
read the trend, not the second decimal. The lesson that does not change: **report per-class recall and macro
metrics** whenever classes are imbalanced — which, in a cell, is always.

✏️ **Your turn.** Add `seed=1` to the four `train_head(...)` calls in 5.2 and re-run. Which conclusions survive a
different seed? Then make the tail even longer (`"4v7r": 3`).

### 📄 Paper connection — imbalanced data (Luo *et al.* 2019)

**Paper**: Luo Z, Zeng X, Bao Z, Xu M. *Deep learning-based strategy for macromolecules classification with imbalanced
data from cellular electron cryotomography.* IJCNN 2019, pp. 1–8. [arXiv:1908.09993](https://arxiv.org/abs/1908.09993)

- **Question it answers:** how to classify subtomograms when some complexes are abundant and others rare.
- **What the paper actually does:** a **multi-path CNN** — one Inception-3D-inspired path per class, each pre-trained as
  a one-vs-rest binary classifier on the imbalanced data, then concatenated with pooling, FC layers and a softmax —
  combined with **mixup** and the **focal loss**; compared with SMOTE, under-sampling, bagging, AdaBoost, genetic
  programming and GSVM-RU. Data: a rat-neuron tomogram with poly-GA aggregates, 40³ voxels at 1.368 nm, classes
  proteasome_d (1,043), proteasome_s (386), TRiC (125), ribosome (80).
- **What our toy keeps:** a long tail (15:1), focal loss, mixup, macro-F1 and G-mean.
- **What our toy leaves out:** the multi-path architecture, end-to-end training on the imbalanced data (we retrain
  only a head on frozen features), real data, the classical resampling baselines.
- **Result reported:** multi-path CNN + mixup + focal loss gave the best Macro-F1 (73.6) and Macro G-mean (84.7), versus
  68.1 / 78.1 for the plain multi-path CNN and 69.5 / 80.3 for GSVM-RU. (The paper's printed G-mean formula appears to
  contain a typo; we use the standard definition above.)
- **Code:** none in AITom.

---
## 6 · Smaller and faster: knowledge distillation (Guo *et al.* 2018)

🧫 At ~1 ms per subtomogram, a CNN processes a million particles in about 20 minutes. A network 3× faster saves hours on
every re-run — and makes CPU-only labs viable. **Knowledge distillation** (Hinton *et al.* 2015) trains a small
**student** network to imitate a large **teacher**. The trick is to imitate the teacher's *soft* outputs at a
**temperature** $T$:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)} .$$

*In words:* dividing the pre-softmax outputs $z$ by $T > 1$ flattens the probability distribution, revealing which
*wrong* classes the teacher considers plausible ("this 70S ribosome looks a little like an 80S ribosome, not at all
like RuBisCO"). That extra information — sometimes called **dark knowledge** — is far richer than a one-hot label.
Guo *et al.* chose $T = 5$. The student is trained with cross-entropy on these soft labels at the same temperature (we
multiply the loss by $T^2$, a standard scaling that keeps gradient sizes comparable).

In [ ]:
# ── 6.1 What the teacher's soft labels look like ──
T_KD = 5.0
teacher = best
z_teacher = predict_logits(teacher, TRAIN["X"])


def softmax(z, T=1.0):
    """Softmax of pre-softmax outputs z at temperature T."""
    e = np.exp((z - z.max(1, keepdims=True)) / T)
    return e / e.sum(1, keepdims=True)


soft_targets = softmax(z_teacher, T_KD)
p1 = softmax(z_teacher, 1.0)
cand = np.where((TRAIN["y"] == 0) & (z_teacher.argmax(1) == 0))[0]            # correctly classified 70S ribosomes
i_ex = int(cand[np.argsort(p1[cand, 0])[len(cand) // 2]])                     # a typical (median-confidence) one
fig, ax = plt.subplots(figsize=(6.4, 2.8))
xs = np.arange(len(CLASSES))
ax.bar(xs - 0.2, softmax(z_teacher[i_ex:i_ex + 1], 1.0)[0], 0.4, label="T = 1 (ordinary softmax)")
ax.bar(xs + 0.2, soft_targets[i_ex], 0.4, label=f"T = {T_KD:.0f} (soft label for the student)")
ax.set_xticks(xs); ax.set_xticklabels(names, fontsize=8); ax.set_ylabel("probability")
ax.set_title(f"Teacher's output for one training subtomogram (true class: {names[0]})"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 At $T = 1$ the teacher puts most of the probability on its top guess; at $T = 5$ the distribution flattens and its
"second opinions" become clearly visible — typically other members of the same biological family.

### 6.2 Train the same tiny student twice: hard labels vs the teacher's soft labels

In [ ]:
# ── 6.2 Distil the teacher into a 4k-weight student ──
class TinyStudent(nn.Module):
    """A deliberately small 3D CNN (≈ 4,000 weights)."""

    def __init__(self, n_classes=6):
        super().__init__()
        self.net = nn.Sequential(*conv_bn_relu(1, 4, stride=2), *conv_bn_relu(4, 8), nn.MaxPool3d(2),
                                 *conv_bn_relu(8, 8), nn.MaxPool3d(2), nn.Flatten(), nn.Linear(8 * 27, n_classes))

    def forward(self, x):
        return self.net(x)


students = {}
for label, targets, T in [("hard labels", one_hot(TRAIN["y"]), 1.0), (f"soft labels (T = {T_KD:.0f})", soft_targets, T_KD)]:
    torch.manual_seed(SEED)
    s = TinyStudent()
    h = train_classifier(s, TRAIN["X"], targets, epochs=6, T=T, verbose=False)
    students[label] = s
    print(f"student, {label:20s}: test acc SNR 0.1 {accuracy(s, TEST_HI['X'], TEST_HI['y']):.2f} | "
          f"SNR 0.03 {accuracy(s, TEST_LO['X'], TEST_LO['y']):.2f}   ({h['seconds']:.0f} s)")


def ms_per_subtomogram(model, X, repeats=3) -> float:
    """Best-of-`repeats` inference time per subtomogram, in milliseconds."""
    best_t = np.inf
    for _ in range(repeats):
        t0 = time.perf_counter(); predict_logits(model, X); best_t = min(best_t, time.perf_counter() - t0)
    return 1000 * best_t / len(X)


X_both = np.concatenate([TEST_HI["X"], TEST_LO["X"]]); y_both = np.concatenate([TEST_HI["y"], TEST_LO["y"]])
kd_rows = [("teacher " + best_name, teacher)] + [("student, " + k, v) for k, v in students.items()]
fig, ax = plt.subplots(figsize=(5.6, 3.4))
for (label, m), c in zip(kd_rows, ["tab:blue", "tab:gray", "tab:green"]):
    ms, acc = ms_per_subtomogram(m, X_both), accuracy(m, X_both, y_both)
    print(f"{label:30s} {n_params(m):7,d} weights  {ms:.3f} ms/subtomogram  test acc (both SNRs) {acc:.2f}")
    ax.scatter(ms, acc, s=40 + n_params(m) / 300, color=c, label=f"{label} ({n_params(m):,} w)")
ax.set_xlabel("inference time (ms per subtomogram, this CPU)"); ax.set_ylabel("test accuracy (both SNRs)")
ax.set_title("Distillation: speed vs accuracy (marker size ∝ #weights)")
ax.set_xlim(0, None); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The student has several times fewer weights than the teacher and runs faster (the exact speed-up
depends on your CPU and on fixed per-batch costs; tiny networks are rarely limited by arithmetic alone). Trained on the
teacher's soft labels, it typically recovers part of the accuracy gap left by training on hard labels: the soft labels
tell it *how* classes resemble each other. Guo *et al.* observed the same direction on a much larger scale (see box).
In a small toy like ours the gap can be small or even flip for one seed — check with ✏️ below.

✏️ **Your turn.** Try `T_KD = 1` and `T_KD = 20` (cell 6.1, then re-run 6.2). Which temperature helps the student
most? What happens to the soft labels as $T \to \infty$?

### 📄 Paper connection — model compression (Guo *et al.* 2018)

**Paper**: Guo J, Zhou B, Zeng X, Freyberg Z, Xu M. *Model compression for faster structural separation of
macromolecules captured by cellular electron cryo-tomography.* ICIAR 2018, LNCS, pp. 144–152.
[arXiv:1801.10597](https://arxiv.org/abs/1801.10597)

- **Question it answers:** can a large subtomogram classifier be made much smaller and faster without losing much
  accuracy?
- **What the paper actually does:** knowledge distillation (teacher → student) combined with hand-simplified student
  architectures (fewer filters, no FC-1024 layer, 3³ instead of 2³ pooling). Students are trained with categorical
  cross-entropy on the teacher's softened outputs, $p_i = \exp(z_i/T)/\sum_j \exp(z_j/T)$ with **$T = 5$** (chosen
  heuristically), and compared with training on hard labels. No pruning or quantisation; the paper does not specify a
  weighting between hard and soft loss terms (our toy uses the soft term only).
- **What our toy keeps:** teacher/student, $T = 5$, soft- vs hard-label comparison, parameter count and timing.
- **What our toy leaves out:** the DSRF3D-v2 teacher (18.3 M weights) and the three student designs; the 23-class
  benchmark at SNR 0.05.
- **Result reported:** teacher 95.89 %; student s1 93.82 % with 18.1× fewer parameters and 1.69× faster; student s3
  89.83 % at 113× compression and 2.85× faster; soft labels beat hard labels for every student (s3: 89.8 % vs 72.2 %).
- **Code:** none in AITom.

---
## 7 · From 2D to 3D: kernel inflation and video-model transfer (Jain, Li & Xu 2024)

🧫 Labelled subtomograms are rare, so 3D networks are usually trained from scratch on small datasets. The world of
**videos**, by contrast, has huge labelled collections (e.g. Kinetics-400 human-action clips), and a video is also a 3D
array: height × width × **time**. Jain, Li & Xu (2024) took networks pre-trained on videos, fed them subtomograms with
the **z axis playing the role of time**, and fine-tuned them — "knowledge transfer from macro-world to micro-world".

How can a 2D image network become a 3D network in the first place? The classic recipe, **kernel inflation** (Carreira &
Zisserman 2017, "I3D"), copies every 2D kernel $W_{2D}$ (size $k \times k$) along a new axis and divides by $k$:

$$W_{3D}[a, i, j] \;=\; \frac{1}{k}\, W_{2D}[i, j], \qquad a = 1, \dots, k .$$

*In words:* the inflated filter first averages $k$ neighbouring z-slices, then applies the 2D filter. Consequence — the
**"boring video" property**: a volume made of the *same* image repeated along z produces exactly the response the 2D
network gives to that single image. Our toy chains the two ideas in miniature:

1. pre-train a small **2D** CNN on a *different task* with plentiful cheap data: classify 2D projection images of six
   *other* zoo molecules (mitoribosome, TRiC, Hsp90, a ubiquitin ligase, an Hsp70 complex, p97);
2. **inflate** it to 3D and verify the boring-video property;
3. **fine-tune** it on our six classes with only 5 or 20 labelled subtomograms per class, versus training from scratch.

(Jain *et al.* did **not** use kernel inflation: they fine-tuned existing video networks. Inflation is our way to
create a "pre-trained 3D network" in seconds.)

In [ ]:
# ── 7.1 A network that can be built in 2D or 3D, and a 2D pre-training task ──
class GAPNet(nn.Module):
    """Small CNN ending in global average pooling; dim=2 for images, dim=3 for volumes (same layer layout)."""

    def __init__(self, n_classes, dim=3, w=8):
        super().__init__()
        Conv, BN, Pool = (nn.Conv2d, nn.BatchNorm2d, nn.MaxPool2d) if dim == 2 else (nn.Conv3d, nn.BatchNorm3d, nn.MaxPool3d)
        pad = dict(padding=1, padding_mode="replicate")          # replicate padding keeps 'boring videos' exact
        self.features = nn.Sequential(Conv(1, w, 3, stride=2, **pad), BN(w), nn.ReLU(),
                                      Conv(w, 2 * w, 3, **pad), BN(2 * w), nn.ReLU(), Pool(2),
                                      Conv(2 * w, 4 * w, 3, **pad), BN(4 * w), nn.ReLU())
        self.head = nn.Linear(4 * w, n_classes)

    def forward(self, x):
        f = self.features(x)
        return self.head(f.mean(dim=tuple(range(2, f.dim()))))   # global average pooling


def simulate_projections(pids, n_per_class, snr, rng):
    """2D images: random 3D rotation → projection along z (sum) → noise → normalise. Returns X [N,1,24,24], y."""
    X, y = [], []
    for k, pid in enumerate(pids):
        for _ in range(n_per_class):
            proj = rotate_volume(TEMPLATES[pid], random_rotation(rng)).sum(0)
            X.append(normalize(add_noise_snr(proj, snr, rng))); y.append(k)
    return np.stack(X)[:, None], np.array(y)


t0 = time.time()
X2d, y2d = simulate_projections(OTHERS_2D, 300, 1.0, np.random.default_rng(SEED + 7))     # cheap, cleaner 2D data
X2d_val, y2d_val = simulate_projections(OTHERS_2D, 40, 1.0, np.random.default_rng(SEED + 8))
net2d = GAPNet(len(OTHERS_2D), dim=2)
torch.manual_seed(SEED)
opt = torch.optim.Adam(net2d.parameters(), lr=3e-3)
Xt2, yt2 = torch.tensor(X2d), torch.tensor(y2d)
for ep in range(12):                                           # 2D training is cheap: plain loop, no augmentation
    net2d.train()
    for i in range(0, len(X2d), 64):
        idx = torch.randperm(len(X2d))[:64]
        loss = F.cross_entropy(net2d(Xt2[idx]), yt2[idx])
        opt.zero_grad(); loss.backward(); opt.step()
net2d.eval()
with torch.no_grad():
    acc2d = (net2d(torch.tensor(X2d_val)).argmax(1).numpy() == y2d_val).mean()
print(f"2D pre-training on {len(X2d)} projections of {len(OTHERS_2D)} OTHER molecules: "
      f"validation accuracy {acc2d:.2f} ({time.time() - t0:.0f} s)")

In [ ]:
# ── 7.2 Inflate 2D kernels to 3D and verify the 'boring video' property ──
def inflate_2d_to_3d(net2d: GAPNet, n_classes: int) -> GAPNet:
    """Copy a 2D GAPNet into a 3D GAPNet: conv kernels repeated along z and divided by k; BN copied; new head."""
    net3d = GAPNet(n_classes, dim=3)
    for m2, m3 in zip(net2d.features, net3d.features):
        if isinstance(m2, nn.Conv2d):
            k = m2.kernel_size[0]
            m3.weight.data = m2.weight.data[:, :, None].repeat(1, 1, k, 1, 1) / k
            m3.bias.data = m2.bias.data.clone()
        elif isinstance(m2, nn.BatchNorm2d):
            m3.load_state_dict(m2.state_dict())
    return net3d


check = inflate_2d_to_3d(net2d, len(OTHERS_2D))
check.head.load_state_dict(net2d.head.state_dict())             # same head, only for this check
check.eval()
img = torch.tensor(X2d_val[:4])                                # 4 images [4, 1, 24, 24]
boring = img[:, :, None].repeat(1, 1, BOX, 1, 1)               # each image repeated 24× along z
with torch.no_grad():
    out2d, out3d = net2d(img), check(boring)
print("2D network on images       :", np.round(out2d[0].numpy(), 3))
print("inflated 3D on boring video:", np.round(out3d[0].numpy(), 3))
print(f"max difference = {(out2d - out3d).abs().max().item():.1e}")
assert torch.allclose(out2d, out3d, atol=1e-4), "inflation should reproduce the 2D outputs on a boring video"

🔍 **What just happened?** The inflated 3D network reproduces the 2D network's outputs to floating-point precision on a
volume that does not change along z. On a real subtomogram, which *does* change along z, the inflated filters start as
"look at a few neighbouring slices, then apply a good 2D filter" — a sensible starting point that fine-tuning then
adapts to genuinely 3D structure.

### 7.3 Fine-tune with few labels: inflated vs random initialisation

In [ ]:
# ── 7.3 Few-label fine-tuning on the six classes: inflated vs from scratch ──
transfer = {}
t0 = time.time()
for n_lab in [5, 20]:
    sel = np.concatenate([np.where(TRAIN["y"] == c)[0][:n_lab] for c in range(len(CLASSES))])
    steps_epochs = int(math.ceil(120 / math.ceil(len(sel) / 16)))   # ≈ 120 gradient steps for every budget
    for init in ["random init", "inflated 2D"]:
        torch.manual_seed(SEED)
        net = GAPNet(len(CLASSES), dim=3) if init == "random init" else inflate_2d_to_3d(net2d, len(CLASSES))
        train_classifier(net, TRAIN["X"][sel], one_hot(TRAIN["y"][sel]), epochs=steps_epochs, batch=16,
                         lr=3e-3, verbose=False)
        transfer[(n_lab, init)] = accuracy(net, X_both, y_both)
        print(f"{n_lab:2d} labels/class, {init:12s}: test accuracy (both SNRs) {transfer[(n_lab, init)]:.2f}")
print(f"({time.time() - t0:.0f} s)")

fig, ax = plt.subplots(figsize=(5, 3))
for j, init in enumerate(["random init", "inflated 2D"]):
    ax.bar(np.arange(2) + (j - 0.5) * 0.35, [transfer[(n, init)] for n in [5, 20]], 0.35, label=init)
ax.axhline(1 / len(CLASSES), color="gray", ls=":", label="chance")
ax.set_xticks(range(2)); ax.set_xticklabels(["5 labels / class", "20 labels / class"])
ax.set_ylabel("test accuracy"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
ax.set_title("Transfer from a 2D task to 3D subtomograms")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With very few labelled subtomograms, starting from the inflated 2D network usually beats
starting from random weights, and the advantage shrinks as labels become more plentiful — the same qualitative pattern
Jain *et al.* report for video-pre-trained networks (largest gains at 5–25 % labelled data). Our "other task" was
deliberately related (2D images of molecules); Jain *et al.*'s source task — human actions in videos — is far more
distant, which makes their result all the more striking. With one seed and tiny test sets, treat our numbers as a
demonstration, not a measurement.

✏️ **Your turn.** Pre-train the 2D network for 1 epoch instead of 8 (cell 7.1) and repeat 7.3. Does a *worse* 2D network
still help? What does that suggest about which pre-trained features transfer?

### 📄 Paper connection — video-model transfer (Jain, Li & Xu 2024)

**Paper**: Jain S, Li X (equal contribution), Xu M. *Knowledge transfer from macro-world to micro-world: enhancing 3D
cryo-ET classification through fine-tuning video-based deep models.* Bioinformatics 40(7):btae368 (2024).
[doi:10.1093/bioinformatics/btae368](https://doi.org/10.1093/bioinformatics/btae368)

- **Question it answers:** can large video models, trained on everyday movies, help 3D cryo-ET classification where
  labelled data are scarce?
- **What the paper actually does:** resizes each subtomogram to 32 × 128 × 128 and treats it as a 32-frame video (first
  axis = time; the single channel is replicated across colour channels, per a figure caption); initialises the backbone
  from a video-pre-trained **3D-ResNet-34 (Kinetics-400)** or **ViViT**, adds a randomly initialised head, and fully
  fine-tunes with $L(\omega) = \frac{1}{|D|}\sum L_{ce}(z(x;\omega), y) + \lambda\lVert\omega\rVert^2$. Video
  initialisation: 30 epochs at lr 1e-4; random initialisation: 100 epochs at 1e-3.
- **What our toy keeps:** a pre-trained network from another domain, a new head, full fine-tuning, the comparison with
  random initialisation at several label budgets.
- **What our toy leaves out:** real video models and data (we inflate a tiny 2D CNN), ViViT, MedMNIST3D, the label
  fractions 5–100 %.
- **Result reported:** video initialisation helped most when labels were scarce: 25 % labelled simulated data
  65.88 % → 81.88 %; 5 % labelled real data (Noble *et al.* single-particle subtomograms, 7 classes) 68.71 % → 87.90 %;
  it also beat self-supervised pre-training (Module 19) and a semi-supervised baseline at low label fractions. On
  MedMNIST3D the gains were mixed.
- **Code:** the paper points to github.com/xulabs/aitom (no specific folder was found).

---
## 8 · Benchmarks and evaluation pitfalls

**Community benchmarks.** Comparing methods fairly requires shared data with ground truth. The SHREC'19 and SHREC'20
"Classification in cryo-electron tomograms" tracks, co-organised with groups including the Xu lab, provided simulated
tomograms of cell-like volumes with 12 protein classes (SHREC'20 also included vesicles and gold fiducials) and scored
*localisation + classification* with precision, recall and F1.

⚠️ **Pitfalls that make published accuracies look better than they are:**

1. **Split by tomogram, not by particle.** Particles from one tomogram share ice thickness, defocus, dose, crowding and
   noise texture. If copies from the same tomogram sit in both training and test sets, the network can recognise the
   *tomogram*, not the molecule (data leakage, Module 05). SHREC'20 kept whole tomograms for testing.
2. **Simulated ≠ real.** A network trained and tested on the same simulator learns its quirks (Module 20).
3. **Accuracy hides rare classes** (Section 5): report per-class recall and macro metrics.
4. **Size matters.** Large complexes are much easier than small ones — SHREC'20 found performance strongly correlated
   with molecular weight. Let us check that in our own results.

In [ ]:
# ── 8.1 Recall vs molecular mass in our classifier ──
pred_lo = predict_logits(best, TEST_LO["X"]).argmax(1)
pred_hi = predict_logits(best, TEST_HI["X"]).argmax(1)
masses = np.array([ZOO[p]["mass_kDa"] for p in CLASSES])
fig, ax = plt.subplots(figsize=(5.4, 3.2))
for lab, pred, d, c in [("SNR 0.1", pred_hi, TEST_HI, "tab:blue"), ("SNR 0.03", pred_lo, TEST_LO, "tab:red")]:
    rec = [(pred[d["y"] == k] == k).mean() for k in range(len(CLASSES))]
    ax.scatter(masses, rec, color=c, label=lab, zorder=3)
    if lab == "SNR 0.1":
        rec_hi = rec
        for m, r, nm in zip(masses, rec, names):
            ax.annotate(nm, (m, r), fontsize=7, xytext=(4, 4), textcoords="offset points")
    else:
        for m, r0, r1 in zip(masses, rec_hi, rec):
            ax.plot([m, m], [r0, r1], color="gray", lw=0.8, zorder=1)
ax.set_xscale("log"); ax.set_xlabel("molecular mass (kDa, log scale)"); ax.set_ylabel("per-class recall")
ax.set_xticks([250, 500, 1000, 2000, 5000]); ax.set_xticklabels(["250", "500", "1000", "2000", "5000"])
ax.minorticks_off()
ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.set_title(f"{best_name}: which molecules are easy?")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In our toy, mass alone does not decide difficulty: RuBisCO (the lightest) is easy because it
has no look-alike among the classes, while the heaviest molecules are confused with their own relatives (70S ↔ 80S).
Recall depends on mass **and** on how many similar molecules compete — in a real cell, with hundreds of species, the
small ones have many look-alikes and also less signal, which is the trend SHREC'20 measured.

### 📄 Paper connection — SHREC'19 and SHREC'20 (Gubins *et al.*)

**Papers**: Gubins I, van der Schot G, Veltkamp RC, Förster F, Du X, Zeng X, Zhu Z, Chang L, Xu M, *et al.*
*Classification in Cryo-Electron Tomograms* (SHREC'19 track), Eurographics Workshop on 3D Object Retrieval 2019,
[doi:10.2312/3dor.20191061](https://doi.org/10.2312/3dor.20191061); Gubins I, Chaillet ML, van der Schot G, …, Zeng X,
Xu M, *et al.* *SHREC 2020: Classification in cryo-electron tomograms.* Computers & Graphics 91:279–289 (2020),
[doi:10.1016/j.cag.2020.07.010](https://doi.org/10.1016/j.cag.2020.07.010) · [arXiv:2203.10035](https://arxiv.org/abs/2203.10035).

- **Question it answers:** how well do localisation + classification methods work on a public, fully annotated
  simulated benchmark?
- **What the benchmarks provide:** SHREC'19 — 10 simulated tomograms with 12 protein types (9 for training, 1 for
  testing); SHREC'20 — 12 protein classes from 42.75 kDa (PDB 1S3X) to 3,325.59 kDa (5MRC) plus vesicles and gold
  fiducials, 1,000–3,000 proteins per volume, 9 tomograms (0–7 train/validation, 8 test), 61 tilts over ±60°, 300 kV,
  SNR 0.12–0.58. Metrics: precision, recall, F1, miss rate, centre distance, multiple hits.
- **Result reported:** SHREC'20 — every learning-based method beat template matching (F1 0.372; 0.576 for a filtered
  variant); DeepFinder had the best overall F1 (0.868); performance correlated strongly with molecular weight (e.g.
  MC DS Net F1 0.594 below 200 kDa vs 0.989 above 600 kDa). SHREC'19: learning-based, especially end-to-end 3D, methods
  performed best. Which entry was the Xu lab's was not verified.

### Where the lab's classifiers live today

| model | what it is | where (AITom) |
|---|---|---|
| DSRF3D, Inception3D | the 2017 subdivision networks (Keras) | `aitom/classify/deep/supervised/cnn/subdivide.py` |
| DSRF3D-v2, CB3D, ResNet3D, DenseNet3D | PyTorch re-implementations (CB3D differs from the 2018 paper) | `aitom/classify/deep/unsupervised/cryo_shift/models/` |
| RB3D, YOPO | listed among AITom's supervised classifiers (YOPO reappears in DISCA, Module 22) | see the AITom README and `doc/tutorials/024_subtomogram_classification` |

**Also from the lab (content not verified):** Uddin MR, Ahmed AY, Khan K, Fatemi MS, Zeng X, Xu M. *Practical analysis
of macromolecule identity from cryo-electron tomography images using deep learning.* IEEE AIPR Workshop 2021, pp. 1–9.
⚠️ We could not access this paper; only its citation is given here. Read it before relying on any detail.

---
## ✅ Checkpoint

1. Three stacked 3 × 3 × 3 convolutions (stride 1) with dilations 1, 2 and 4: what is the receptive field, and how many
   weights does each layer have for one input and one output channel?
   <details><summary>Answer</summary>RF = 1 + 2·(1 + 2 + 4) = 15 voxels; each layer still has 27 weights (+1 bias). Dilation spreads the taps, it does not add weights.</details>
2. Why does the subdivision approach cluster the **pre-softmax** outputs rather than the softmax probabilities?
   <details><summary>Answer</summary>The softmax squashes the outputs onto a simplex and saturates (probabilities near 0 or 1), erasing the graded similarities to each training class. The pre-softmax values keep that profile, which is what lets even an unseen molecule (ClpB) form its own cluster.</details>
3. A classifier reaches 85 % accuracy on an imbalanced test set but never predicts the rarest class. Which metrics
   expose the problem?
   <details><summary>Answer</summary>Per-class recall (0 for the rare class), macro-F1 (lowered by that class's F1 = 0) and the G-mean (exactly 0 when any recall is 0).</details>
4. What does a temperature $T = 5$ do to a teacher's outputs, and why does that help a student?
   <details><summary>Answer</summary>Dividing logits by T > 1 flattens the softmax, so the relative probabilities of the wrong classes become visible ("dark knowledge"); the student learns which classes resemble each other, a richer signal than one-hot labels.</details>
5. In kernel inflation, why is the 2D kernel divided by $k$ when copied along the new axis?
   <details><summary>Answer</summary>So that a volume that is constant along z (a "boring video") gives exactly the same response as the 2D network on one slice: the k copies sum to the original kernel.</details>
6. Why must cryo-ET training and test sets be split by tomogram?
   <details><summary>Answer</summary>Particles from the same tomogram share imaging conditions (defocus, ice thickness, dose, crowding, noise texture); splitting by particle leaks this information and inflates test accuracy.</details>

## 🧠 Questions & discussion

1. The CNN of Section 4 was trained on six molecules. Would subdivision still work for a molecule that is *very*
   different from all six (e.g. a long filament)? What would its pre-softmax profile look like?
2. We augmented only with 180° rotations. Real subtomograms of one molecule appear in *all* orientations anyway. Why is
   augmentation still useful, and what other augmentations would be physically realistic (think CTF, dose, shifts)?
3. Which is worse for a cell biologist: a classifier that misses 50 % of a rare complex, or one that mislabels 5 % of
   ribosomes as that complex? How should this change the loss or the decision threshold?
4. Distillation makes inference cheaper. When would you instead spend *more* compute per subtomogram?
5. Jain *et al.* transferred knowledge from videos of human actions. Propose a source domain that you think would
   transfer *better* to cryo-ET, and how you would test your hypothesis.

## 🚀 Bonus — going further

- **Dilated vs ordinary.** Build a `DSRF3DLite` variant whose middle convolution has `dilation=2` (and `padding=2`),
  train it with `train_classifier`, and compare accuracy and receptive field.
- **Deeper is not always better.** Add a third bottleneck block to `RB3DLite` and a 1 × 2 × 2 first pooling as in CB3D;
  compare at SNR 0.03 with more epochs.
- **Real benchmark data.** Download the SHREC'20 dataset (linked from the arXiv paper) and train a classifier on
  subtomograms cut from tomograms 0–7; test on tomogram 8 — a real split by tomogram.
- **Official code.** Read `aitom/classify/deep/supervised/cnn/subdivide.py` and the AITom tutorial
  `doc/tutorials/024_subtomogram_classification`; map each layer to our lite versions.
- **Deep dive.** The repository's [`SubtomogramClassification`](../../SubtomogramClassification/) tutorial trains
  classifiers on more realistic data.

## 📝 Lab note

```text
Module 16 — date:
Input: simulated 24³ subtomograms (15 Å voxels) of 6 zoo molecules, SNR 0.1 / 0.03, ±60° wedge; ClpB as unseen class
Output: trained DSRF3D-lite / RB3D-lite; subdivision clusters and averages; imbalance, distillation, transfer results
Method: 3D CNN + wedge-safe augmentation; pre-softmax features + k-means; weighted/focal/mixup losses; KD (T = 5);
        2D→3D kernel inflation
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Xu 2017; Che 2018; Luo 2019; Guo 2018; Jain 2024; Gao 2020/21; SHREC'19/'20
```

## 📚 References

1. Xu M, Chai X, Muthakana H, Liang X, Yang G, Zeev-Ben-Mordehai T, Xing E. Deep learning based subdivision approach for large scale macromolecules structure recovery from electron cryo tomograms. *Bioinformatics* 33(14):i13–i22 (2017). doi:10.1093/bioinformatics/btx230
2. Che C, Lin R, Zeng X, Elmaaroufi K, Galeotti J, Xu M. Improved deep learning-based macromolecules structure classification from electron cryo-tomograms. *Mach Vis Appl* 29(8):1227–1236 (2018). doi:10.1007/s00138-018-0949-4
3. Gao S, Han R, Zeng X, Cui X, Liu Z, Xu M, Zhang F. Dilated-DenseNet for macromolecule classification in cryo-electron tomography. ISBRA 2020, LNBI 12304:82–94. doi:10.1007/978-3-030-57821-3_8
4. Gao S, Han R, Zeng X, Liu Z, Xu M, Zhang F. Macromolecules structural classification with a 3D dilated dense network in cryo-electron tomography. *IEEE/ACM TCBB* (2021). doi:10.1109/TCBB.2021.3065986
5. Luo Z, Zeng X, Bao Z, Xu M. Deep learning-based strategy for macromolecules classification with imbalanced data from cellular electron cryotomography. IJCNN 2019, pp. 1–8. arXiv:1908.09993
6. Guo J, Zhou B, Zeng X, Freyberg Z, Xu M. Model compression for faster structural separation of macromolecules captured by cellular electron cryo-tomography. ICIAR 2018, LNCS, pp. 144–152. arXiv:1801.10597
7. Jain S, Li X, Xu M. Knowledge transfer from macro-world to micro-world: enhancing 3D cryo-ET classification through fine-tuning video-based deep models. *Bioinformatics* 40(7):btae368 (2024). doi:10.1093/bioinformatics/btae368
8. Gubins I *et al.* Classification in cryo-electron tomograms (SHREC'19). Eurographics 3DOR 2019. doi:10.2312/3dor.20191061
9. Gubins I *et al.* SHREC 2020: Classification in cryo-electron tomograms. *Comput Graph* 91:279–289 (2020). doi:10.1016/j.cag.2020.07.010
10. Uddin MR, Ahmed AY, Khan K, Fatemi MS, Zeng X, Xu M. Practical analysis of macromolecule identity from cryo-electron tomography images using deep learning. IEEE AIPR 2021, pp. 1–9. *(content not verified)*
11. Hinton G, Vinyals O, Dean J. Distilling the knowledge in a neural network. arXiv:1503.02531 (2015).
12. Lin TY, Goyal P, Girshick R, He K, Dollár P. Focal loss for dense object detection. ICCV 2017. doi:10.1109/ICCV.2017.324
13. Zhang H, Cisse M, Dauphin YN, Lopez-Paz D. mixup: Beyond empirical risk minimization. ICLR 2018. arXiv:1710.09412
14. Carreira J, Zisserman A. Quo vadis, action recognition? A new model and the Kinetics dataset. CVPR 2017. arXiv:1705.07750
15. Zeng X, Xu M. AITom: Open-source AI platform for cryo-electron tomography data analysis. arXiv:1911.03044 (2019).

## ➡️ Next

**[17 · Looking inside networks and segmenting structures voxel by voxel](17_interpretability_segmentation.ipynb)** —
you will ask *why* the classifier decides as it does (Respond-CAM) and teach networks to outline molecules voxel by
voxel. Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).